In [1]:
from albumentations import ToTensorV2
from torch import optim
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import torch
import random
import numpy as np
import torch.nn as nn
import albumentations as Albu
import pandas as pd
from torch.utils.data.sampler import RandomSampler
from warmup_scheduler import GradualWarmupScheduler
import os

from utils.dataset import PandasDataset
from utils.metrics import model_checkpoint
from utils.train import train_model
from utils.models import EfficientNetApi

In [2]:
seed = 10
shuffle = True
batch_size = 6
num_workers = 4
output_classes = 5
init_lr = 3e-4
warmup_factor = 2
warmup_epochs = 1
n_epochs = 50
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
loss_function = nn.BCEWithLogitsLoss()

torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)

ROOT_DIR = '../../..'

data_dir = '../../../..'
images_dir = os.path.join(data_dir, 'tiles')

Using device: cuda


In [3]:
load_model = efficientnet_b0(
     weights=EfficientNet_B0_Weights.DEFAULT
)
model = EfficientNetApi(model=load_model, output_dimensions=output_classes, dropout_rate=0.6)
model = model.to(device)

In [4]:
print("Using device:", device)
loss_function = nn.BCEWithLogitsLoss()

torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)

Using device: cuda


In [5]:
from sklearn.model_selection import train_test_split

df_train_ = pd.read_csv(f"{ROOT_DIR}/data/train_5fold.csv")
print(f"Registros originais: {len(df_train_)}")

df_entropy = pd.read_csv(f"{ROOT_DIR}/data/entropy.csv")

df_entropy_sorted = df_entropy.sort_values(by="difficulty_score", ascending=False).reset_index(drop=True)

n_remove = int(len(df_entropy_sorted) * 0.2)
df_entropy_top50 = df_entropy_sorted.head(n_remove)


df_train_ = df_train_[~df_train_['image_id'].isin(df_entropy_top50['image_id'])].reset_index(drop=True)

print(f"Registros no df_entropy: {len(df_entropy)}")
print(f"Registros no df_entropy_top50: {len(df_entropy_top50)}")
print(f"Registros filtrados (df_train_): {len(df_train_)}")

df_train, df_val = train_test_split(df_train_, test_size=0.2, random_state=seed)

df_test = pd.read_csv(f"{ROOT_DIR}/data/test.csv")

def remove_nonexistent_images(df, images_dir):
    """
    Remove rows from df where the image does not exist in images_dir.
    """
    image_ids = df['image_id'].apply(lambda x: os.path.join(images_dir, f"{x}.png"))
    existent_images = [os.path.isfile(path) for path in image_ids]
    df = df[existent_images]
    return df

df_train = remove_nonexistent_images(df_train, images_dir)
df_val = remove_nonexistent_images(df_val, images_dir)
df_test = remove_nonexistent_images(df_test, images_dir)

print(f"\nTreino: {len(df_train)} registros")
print(f"Validação: {len(df_val)} registros")
print(f"Teste: {len(df_test)} registros")

Registros originais: 9024
Registros no df_entropy: 903
Registros no df_entropy_top50: 180
Registros filtrados (df_train_): 8844

Treino: 7072 registros
Validação: 1768 registros
Teste: 1590 registros


#### view data

In [6]:
(df_train.shape, df_val.shape, df_test.shape)

((7072, 5), (1768, 5), (1590, 4))

In [7]:
from utils.dataset import RGB2XYZTransform

transforms = Albu.Compose([
    Albu.Transpose(p=0.5),
    Albu.VerticalFlip(p=0.5),
    Albu.HorizontalFlip(p=0.5),
    # ToTensorV2()
])

valid_transforms = Albu.Compose([
    # RGB2XYZTransform(p=1.0),
    # ToTensorV2()
])

In [8]:
df_train.columns = df_train.columns.str.strip()

train_dataset = PandasDataset(images_dir, df_train, transforms=transforms, format="png")
valid_dataset = PandasDataset(images_dir, df_val, transforms=valid_transforms, format="png")
test_dataset = PandasDataset(images_dir, df_test, transforms=valid_transforms, format="png")

In [9]:
train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=batch_size, num_workers=num_workers, sampler=RandomSampler(train_dataset)
)
valid_loader = torch.utils.data.DataLoader(
    valid_dataset, batch_size=batch_size, num_workers=num_workers, sampler = RandomSampler(valid_dataset)
)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=batch_size, num_workers=num_workers, sampler = RandomSampler(test_dataset)
)

In [10]:
optimizer = optim.Adam(model.parameters(), lr = init_lr / warmup_factor)
scheduler_cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, n_epochs - warmup_epochs)
scheduler = GradualWarmupScheduler(optimizer, multiplier = warmup_factor, total_epoch = warmup_epochs, after_scheduler=scheduler_cosine)

In [ ]:
train_model(
    model=model,
    epochs=n_epochs,
    optimizer=optimizer,
    scheduler=scheduler,
    train_dataloader=train_loader,
    valid_dataloader=valid_loader,
    checkpoint=model_checkpoint,
    device=device,
    loss_function=loss_function,
    path_to_save_metrics="logs/b0-entropy-png.txt",
    path_to_save_model="models/b0-entropy-png.pth",
    patience=5,
)

Epoch 1/50



loss: 0.64804, smooth loss: 0.62753:   3%|▎         | 30/1179 [00:19<11:33,  1.66it/s]

# tests

In [10]:
from utils.metrics import evaluation, format_metrics
model.load_state_dict(
    torch.load(f"models/b0-entropy-png.pth")
)
response = evaluation(model, test_loader, device)
result = format_metrics(response[0])
print(result)

100%|██████████| 265/265 [01:22<00:00,  3.21it/s]


VAL_ACC      Mean: 62.790 | Std: 1.260 | 95% CI: [60.692, 64.780]
VAL_KAPPA    Mean: 0.838 | Std: 0.012 | 95% CI: [0.818, 0.858]
VAL_F1       Mean: 0.564 | Std: 0.013 | 95% CI: [0.543, 0.585]
VAL_RECALL   Mean: 0.569 | Std: 0.013 | 95% CI: [0.547, 0.590]
VAL_PRECISION Mean: 0.566 | Std: 0.013 | 95% CI: [0.545, 0.587]


In [13]:
import matplotlib.pyplot as plt
import numpy as np

# sua matriz (exemplo)
cm = np.array(response[0].get("confusion_matrix"))
print(cm)
#
# classes = [0, 1, 2, 3, 4, 5]
#
# plt.figure(figsize=(8, 6))
# im = plt.imshow(cm, interpolation='nearest', cmap='Blues')
# plt.title("Matriz de Confusão (Normalizada) - Entropia")
# plt.colorbar(im, fraction=0.046, pad=0.04)
#
# tick_marks = np.arange(len(classes))
# plt.xticks(tick_marks, classes)
# plt.yticks(tick_marks, classes)
#
# plt.ylabel("Classe Verdadeira")
# plt.xlabel("Classe Predita")
#
# # adicionar valores nas células
# fmt = ".3f"
# thresh = cm.max() / 2.
# for i in range(cm.shape[0]):
#     for j in range(cm.shape[1]):
#         plt.text(j, i, format(cm[i, j], fmt),
#                  ha="center", va="center",
#                  color="white" if cm[i, j] > thresh else "black")
#
# plt.tight_layout()
# plt.show()


[[0.87121937 0.08766134 0.01591455 0.00674685 0.01615625 0.00230163]
 [0.11996772 0.66333769 0.19687277 0.01982182 0.         0.        ]
 [0.03001837 0.19950503 0.5296471  0.19538755 0.03057903 0.01486293]
 [0.03798319 0.03266268 0.25268286 0.32434794 0.20615627 0.14616706]
 [0.05871018 0.04738143 0.07413365 0.15678768 0.38516844 0.27781861]
 [0.03190994 0.01083768 0.0160288  0.10286192 0.20107789 0.63728377]]
